# Exploratory Data Analysis (EDA)


### Used libraries:

In [23]:
import polars as pl
import kagglehub
from pathlib import Path
import shutil
from tqdm import tqdm

### 1. Data ingestion

In [ ]:
dataset = "elemento/nyc-yellow-taxi-trip-data"
file = "yellow_tripdata_2015-01.csv"
bronze_path = Path().cwd().parent / "data" / "bronze" / file

if not bronze_path.exists():
    dataset_path = Path(kagglehub.dataset_download(dataset, file))
    shutil.copy(dataset_path, bronze_path)

In [9]:
lf = pl.scan_csv(bronze_path)
lf.head(10).collect()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
i64,str,str,i64,f64,f64,f64,i64,str,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64
2,"""2015-01-15 19:05:39""","""2015-01-15 19:23:42""",1,1.59,-73.993896,40.750111,1,"""N""",-73.974785,40.750618,1,12.0,1.0,0.5,3.25,0.0,0.3,17.05
1,"""2015-01-10 20:33:38""","""2015-01-10 20:53:28""",1,3.3,-74.001648,40.724243,1,"""N""",-73.994415,40.759109,1,14.5,0.5,0.5,2.0,0.0,0.3,17.8
1,"""2015-01-10 20:33:38""","""2015-01-10 20:43:41""",1,1.8,-73.963341,40.802788,1,"""N""",-73.95182,40.824413,2,9.5,0.5,0.5,0.0,0.0,0.3,10.8
1,"""2015-01-10 20:33:39""","""2015-01-10 20:35:31""",1,0.5,-74.009087,40.713818,1,"""N""",-74.004326,40.719986,2,3.5,0.5,0.5,0.0,0.0,0.3,4.8
1,"""2015-01-10 20:33:39""","""2015-01-10 20:52:58""",1,3.0,-73.971176,40.762428,1,"""N""",-74.004181,40.742653,2,15.0,0.5,0.5,0.0,0.0,0.3,16.3
1,"""2015-01-10 20:33:39""","""2015-01-10 20:53:52""",1,9.0,-73.874374,40.774048,1,"""N""",-73.986977,40.758194,1,27.0,0.5,0.5,6.7,5.33,0.3,40.33
1,"""2015-01-10 20:33:39""","""2015-01-10 20:58:31""",1,2.2,-73.983276,40.726009,1,"""N""",-73.99247,40.749634,2,14.0,0.5,0.5,0.0,0.0,0.3,15.3
1,"""2015-01-10 20:33:39""","""2015-01-10 20:42:20""",3,0.8,-74.002663,40.734142,1,"""N""",-73.99501,40.726326,1,7.0,0.5,0.5,1.66,0.0,0.3,9.96
1,"""2015-01-10 20:33:39""","""2015-01-10 21:11:35""",3,18.2,-73.783043,40.644356,2,"""N""",-73.987595,40.759357,2,52.0,0.0,0.5,0.0,5.33,0.3,58.13


### 2. Initial diagnosis

In [12]:
numeric_columns = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount"
]

lf.select(numeric_columns).describe()

statistic,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",1.2748986e7,1.2748986e7,1.2748986e7,1.2748986e7,1.2748986e7,1.2748986e7,1.2748986e7,1.2748983e7,1.2748986e7
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0
"""mean""",1.681491,13.45913,11.905659,0.308279,0.497799,1.853814,0.243498,0.283143,15.108295
"""std""",1.337924,9844.094218,10.302537,0.591664,0.035342,1106.432314,1.527171,0.069086,1106.503247
"""min""",0.0,0.0,-450.0,-79.0,-0.5,-92.42,-26.0,0.0,-450.3
"""25%""",1.0,1.0,6.5,0.0,0.5,0.0,0.0,0.3,8.16
"""50%""",1.0,1.68,9.0,0.0,0.5,1.0,0.0,0.3,11.16
"""75%""",2.0,3.0,13.5,0.5,0.5,2.06,0.0,0.3,16.3
"""max""",9.0,1.5420e7,4008.0,999.99,0.5,3950588.8,1450.09,0.3,3950611.6


### 3. Data transformation

In [39]:
df_silver = (
    lf.filter(pl.col("fare_amount") >= 0)
    .filter(pl.col("extra") >= 0)
    .filter(pl.col("mta_tax") >= 0)
    .filter(pl.col("tip_amount") >= 0)
    .filter(pl.col("tolls_amount") >= 0)
    .filter(pl.col("total_amount") >= 0)
    .with_columns(
        pl.col("tpep_pickup_datetime").str.to_datetime().alias("tpep_pickup_datetime"),
        pl.col("tpep_dropoff_datetime").str.to_datetime().alias("tpep_dropoff_datetime")
    )
    .rename({
        "VendorID": "vendor_id",
        "RateCodeID": "rate_code_id"
    })
    .collect()
)

bronze_size = lf.select(pl.len()).collect().item()
silver_size = df_silver.shape[0]

print(f"Bronze size: {bronze_size}")
print(f"Silver size: {silver_size}")
print(f"Rows affected: {bronze_size - silver_size}")

Bronze size: 12748986
Silver size: 12744911
Rows affected: 4075


In [40]:
df_silver.head()

vendor_id,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,rate_code_id,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
i64,datetime[μs],datetime[μs],i64,f64,f64,f64,i64,str,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64
2,2015-01-15 19:05:39,2015-01-15 19:23:42,1,1.59,-73.993896,40.750111,1,"""N""",-73.974785,40.750618,1,12.0,1.0,0.5,3.25,0.0,0.3,17.05
1,2015-01-10 20:33:38,2015-01-10 20:53:28,1,3.3,-74.001648,40.724243,1,"""N""",-73.994415,40.759109,1,14.5,0.5,0.5,2.0,0.0,0.3,17.8
1,2015-01-10 20:33:38,2015-01-10 20:43:41,1,1.8,-73.963341,40.802788,1,"""N""",-73.95182,40.824413,2,9.5,0.5,0.5,0.0,0.0,0.3,10.8
1,2015-01-10 20:33:39,2015-01-10 20:35:31,1,0.5,-74.009087,40.713818,1,"""N""",-74.004326,40.719986,2,3.5,0.5,0.5,0.0,0.0,0.3,4.8
1,2015-01-10 20:33:39,2015-01-10 20:52:58,1,3.0,-73.971176,40.762428,1,"""N""",-74.004181,40.742653,2,15.0,0.5,0.5,0.0,0.0,0.3,16.3


### 4. Silver data loading

In [38]:
silver_path = Path().cwd().parent / "data" / "silver" / bronze_path.with_suffix(".parquet").name

df_silver.write_parquet(silver_path)